# Chapter 7 — Examples Are Experimental Data

**Book alignment:** DSPy From First Principles, Chapter 7

**Question this notebook isolates:** Does the fixture enforce family-based splits and input/evaluation boundaries that a bag-of-rows dataset would silently violate?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from common.data import canonical_split, teaching_cases
from common.dspy_program import to_dspy_example


## Split by family, not by row

Random row splits leak near-siblings across boundaries — the program looks like it learned to edit when it learned an author's cadence. All four `dialogue` cases sit in dev; all four `technical-explanation` cases sit in holdout. No family straddles a boundary.


In [ ]:
split = canonical_split()
print(f"train: {len(split.train)}, dev: {len(split.dev)}, holdout: {len(split.holdout)}")
print(f"train ids: {list(split.train_ids)[:4]}...")
print(f"dev ids:   {list(split.dev_ids)}")
print(f"holdout:   {list(split.holdout_ids)}")

CANONICAL_ROLES = {"ed-001": "train", "ed-002": "train", "ed-003": "dev", "ed-004": "holdout"}
role_of = {}
for role, ids in (("train", split.train_ids), ("dev", split.dev_ids), ("holdout", split.holdout_ids)):
    for cid in ids:
        role_of[cid] = role
print("canonical roles:", {cid: role_of[cid] for cid in CANONICAL_ROLES})


In [ ]:
assert (len(split.train), len(split.dev), len(split.holdout)) == (26, 11, 7)
assert all(role_of[cid] == role for cid, role in CANONICAL_ROLES.items())
assert set(split.train_ids).isdisjoint(split.dev_ids)
assert set(split.train_ids).isdisjoint(split.holdout_ids)
assert set(split.dev_ids).isdisjoint(split.holdout_ids)
train_groups = {c.source_group for c in split.train}
dev_groups = {c.source_group for c in split.dev}
holdout_groups = {c.source_group for c in split.holdout}
assert train_groups.isdisjoint(dev_groups)
assert train_groups.isdisjoint(holdout_groups)
assert dev_groups.isdisjoint(holdout_groups)
print("26/11/7 with disjoint families; canonical cases kept their roles")


## Program inputs exclude the answers

The `.with_inputs` projection is the machine-checkable form of Chapter 3's third design question — could this value contain, imply, or be derived from the answer? Anything answer-derived that crosses into generation is contamination in the direction that looks like success.


In [ ]:
cases = {c.case_id: c for c in teaching_cases()}
example = to_dspy_example(cases["ed-003"])
program_inputs = dict(example.inputs())
EVAL_FIELDS = ("required_entities", "forbidden_terms", "semantic_constraints", "reference_rewrite")
print("program inputs:", sorted(program_inputs))
print("eval-side only:", [f for f in EVAL_FIELDS if hasattr(example, f)])


In [ ]:
assert set(program_inputs) == {"sentence", "goal", "context"}
assert all(f not in program_inputs for f in EVAL_FIELDS)
assert all(hasattr(example, f) for f in EVAL_FIELDS)
print("reference and constraints attached for evaluation, invisible to generation")


## A fixture has computable granularity

With 11 development cases, one case owns about 9% of the mean — before any model nondeterminism enters. A single-run `+0.011` is therefore not evidence of improvement; it is one modest case movement or execution-state variation away from zero. Restraint cases keep the corpus from teaching that change is always good.


In [ ]:
all_cases = teaching_cases()
n_dev = len(canonical_split().dev)
one_case_weight = 1 / n_dev
swing_035 = 0.35 / n_dev
swing_011 = 0.11 / n_dev
restraint = [c.case_id for c in all_cases if c.target_failure == "unnecessary_edit"]
print(f"dev cases: {n_dev}, one-case weight: {one_case_weight:.4f}")
print(f"0.35 case swing moves dev mean by: {swing_035:.4f}")
print(f"0.11 case swing moves dev mean by: {swing_011:.4f}")
print(f"restraint cases ({len(restraint)}): {restraint}")


In [ ]:
assert n_dev == 11
assert abs(one_case_weight - 0.0909) < 1e-3
assert abs(swing_035 - 0.0318) < 1e-3
assert abs(swing_011 - 0.010) < 1e-4
assert len(restraint) == 6
print("granularity computed before the experiment, not excused after it")


## What we earned

The fixture is an instrument with known resolution: 44 cases across 12 families, split at family boundaries, fingerprinted twice, with failure-mode coverage and restraint built in rather than sampled by accident.

Notebook 08 / Chapter 8 spends the apparatus for the first time — a frozen development baseline that turns rewrites and rows into numbers. That function will prove more consequential than the program: what exactly are we measuring, and what happens when the measurement is wrong?
